# Unemployment Rate — Silver
2021–2025 | Regional unemployment rates (%) | Initial load only.

Adapted from `grdp_per_capita.ipynb`. Source: `pondolens.dev_bronze.unemployment_rate`; destination: `pondolens.dev_silver.unemployment_rate`. **Run cells in Databricks SQL mode** (set notebook default language to SQL, or change code cell language to SQL).

**Scope:** Unemployment rate only, not number of unemployed persons. One record per year and region. The supplied source workbook contains overlapping table versions and estimate statuses; select one authoritative record per business key, with final estimates preferred, then newest release. The eight requested output columns cannot preserve source version and status; review conflicts before creating Silver.
**Year fix:** Bronze stores year as strings such as `2021.0`. All year transformations and filters use a two-stage DECIMAL-to-INT conversion. The Silver build uses `CREATE OR REPLACE TABLE` so an existing empty table is rebuilt.


## 1. Source coverage and schema
Inspect Bronze fields, years, geography, indicators, and estimate status. The workbook has `region`, `geographic_name`, `geographic_level`, `indicator`, `value`, `estimate_status`, and `table_version`. If your ingested Bronze table renamed these fields, update the normalization SQL below before running.

In [0]:
DESCRIBE TABLE pondolens.dev_bronze.unemployment_rate;

In [0]:
SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year, geographic_level, indicator, estimate_status,
       COUNT(*) AS total_rows,
       COUNT_IF(TRY_CAST(REGEXP_REPLACE(CAST(value AS STRING), ',', '') AS DECIMAL(18,6)) IS NULL) AS non_numeric_values
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY 1,2,3,4 ORDER BY 1,2,3,4;

## 2. Review repeated business keys and conflicting estimates
Only `geographic_level = region` and `indicator = Unemployment rate` are in scope. Repeated region-year rows can reflect distinct releases, preliminary/final estimates, or true duplicates. Review before applying the selection rule.

In [0]:
SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year, UPPER(TRIM(geographic_name)) AS region_name,
       COUNT(*) AS record_count, COUNT(DISTINCT CAST(value AS STRING)) AS distinct_values,
       COLLECT_SET(CONCAT(COALESCE(estimate_status,'NULL'), ' | ', COALESCE(table_version,'NULL'), ' | ', COALESCE(CAST(value AS STRING),'NULL'))) AS source_versions
FROM pondolens.dev_bronze.unemployment_rate
WHERE LOWER(TRIM(geographic_level)) = 'region'
  AND LOWER(TRIM(indicator)) = 'unemployment rate'
GROUP BY 1,2 HAVING COUNT(*) > 1
ORDER BY year, region_name;

## 3. Create table and clean
Standardize names and assign regional codes using an explicit mapping. Reject invalid rates and unmapped regions for manual review rather than silently inserting them. Rank duplicate source versions: final before preliminary, then most recent year mentioned in `table_version`, then non-null values. **If multiple values remain tied at the top rank, resolve them before accepting the output.** IDs start at 100000001, matching the GRDP notebook convention. `run_id` identifies this Silver build and `source` records Bronze provenance. `CREATE TABLE IF NOT EXISTS` means rerunning does not overwrite an existing table.

## Region mapping correction
Use the Bronze `region` label first, because `geographic_name` can be blank, generic, or inconsistent. The previous mapping also had overlapping Roman-numeral patterns (e.g. Region IX and Region I). Validate coverage before and after rebuilding.


In [0]:
SELECT region, geographic_name, year, COUNT(*) AS rows
FROM pondolens.dev_bronze.unemployment_rate
WHERE LOWER(TRIM(geographic_level)) = 'region'
  AND LOWER(TRIM(indicator)) = 'unemployment rate'
GROUP BY region, geographic_name, year
ORDER BY region, year;


In [0]:
CREATE OR REPLACE TABLE pondolens.dev_silver.unemployment_rate
USING DELTA AS
WITH normalized AS (
 SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year,
        UPPER(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(region), ''), geographic_name), '\\s+', ' '))) AS raw_region_name,
        TRY_CAST(REGEXP_REPLACE(TRIM(CAST(value AS STRING)), ',', '') AS DECIMAL(12,4)) AS unemployment_rate,
        LOWER(TRIM(estimate_status)) AS estimate_status,
        COALESCE(table_version, '') AS table_version
 FROM pondolens.dev_bronze.unemployment_rate
 WHERE TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) BETWEEN 2021 AND 2025
   AND LOWER(TRIM(geographic_level)) = 'region'
   AND LOWER(TRIM(indicator)) = 'unemployment rate'
), mapped AS (
 SELECT *,
   CASE
     WHEN raw_region_name RLIKE 'NATIONAL CAPITAL REGION|\\(NCR\\)' THEN 'NCR'
     WHEN raw_region_name RLIKE 'CORDILLERA ADMINISTRATIVE REGION|\\(CAR\\)' THEN 'CAR'
     WHEN raw_region_name RLIKE 'ILOCOS|REGION I \\(' THEN 'I'
     WHEN raw_region_name RLIKE 'CAGAYAN VALLEY|REGION II \\(' THEN 'II'
     WHEN raw_region_name RLIKE 'CENTRAL LUZON|REGION III \\(' THEN 'III'
     WHEN raw_region_name RLIKE 'CALABARZON|REGION IV-A|REGION IVA' THEN 'IV-A'
     WHEN raw_region_name RLIKE 'MIMAROPA|REGION IV-B|REGION IVB' THEN 'IV-B'
     WHEN raw_region_name RLIKE 'BICOL|REGION V \\(' THEN 'V'
     WHEN raw_region_name RLIKE 'WESTERN VISAYAS|REGION VI \\(' THEN 'VI'
     WHEN raw_region_name RLIKE 'CENTRAL VISAYAS|REGION VII \\(' THEN 'VII'
     WHEN raw_region_name RLIKE 'EASTERN VISAYAS|REGION VIII \\(' THEN 'VIII'
     WHEN raw_region_name RLIKE 'ZAMBOANGA PENINSULA|REGION IX \\(' THEN 'IX'
     WHEN raw_region_name RLIKE 'NORTHERN MINDANAO|REGION X \\(' THEN 'X'
     WHEN raw_region_name RLIKE 'DAVAO|REGION XI \\(' THEN 'XI'
     WHEN raw_region_name RLIKE 'SOCCSKSARGEN|REGION XII \\(' THEN 'XII'
     WHEN raw_region_name RLIKE 'CARAGA|REGION XIII \\(' THEN 'XIII'
     WHEN raw_region_name RLIKE 'BANGSAMORO|BARMM|ARMM|AUTONOMOUS REGION IN MUSLIM MINDANAO' THEN 'BARMM'
     WHEN raw_region_name RLIKE 'NEGROS ISLAND REGION|\\(NIR\\)' THEN 'NIR'
   END AS region_code
 FROM normalized
), valid AS (
 SELECT *,
 CASE region_code
 WHEN 'NCR' THEN 'National Capital Region' WHEN 'CAR' THEN 'Cordillera Administrative Region'
 WHEN 'I' THEN 'Ilocos Region' WHEN 'II' THEN 'Cagayan Valley' WHEN 'III' THEN 'Central Luzon'
 WHEN 'IV-A' THEN 'CALABARZON' WHEN 'IV-B' THEN 'MIMAROPA' WHEN 'V' THEN 'Bicol Region'
 WHEN 'VI' THEN 'Western Visayas' WHEN 'VII' THEN 'Central Visayas' WHEN 'VIII' THEN 'Eastern Visayas'
 WHEN 'IX' THEN 'Zamboanga Peninsula' WHEN 'X' THEN 'Northern Mindanao' WHEN 'XI' THEN 'Davao Region'
 WHEN 'XII' THEN 'SOCCSKSARGEN' WHEN 'XIII' THEN 'Caraga' WHEN 'BARMM' THEN 'BARMM'
 WHEN 'NIR' THEN 'Negros Island Region' END AS region_name,
 COALESCE(TRY_CAST(REGEXP_EXTRACT(table_version, '(20[0-9]{2})(?!.*20[0-9]{2})', 1) AS INT), 0) AS release_year
 FROM mapped WHERE region_code IS NOT NULL AND unemployment_rate BETWEEN 0 AND 100
), ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY year, region_code
 ORDER BY CASE estimate_status WHEN 'final' THEN 0 WHEN 'preliminary' THEN 1 ELSE 2 END,
 release_year DESC, unemployment_rate DESC, table_version DESC) AS selection_rank
 FROM valid
)
SELECT CAST(100000000 + ROW_NUMBER() OVER (ORDER BY year, region_code) AS INT) AS unemployment_rate_id,
 year, region_code, region_name, unemployment_rate,
 CONCAT('silver_unemployment_', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')) AS run_id,
 'pondolens.dev_bronze.unemployment_rate' AS source,
 CURRENT_TIMESTAMP() AS processed_at
FROM ranked WHERE selection_rank = 1;

## 4. Review data and types

In [0]:
SELECT * FROM pondolens.dev_silver.unemployment_rate ORDER BY year, region_code;
DESCRIBE TABLE pondolens.dev_silver.unemployment_rate;

## 5. Reconcile source and Silver counts
Bronze in-scope rows may exceed Silver rows because multiple table releases cover the same region-year. Compare **distinct region-year keys**, not raw row counts. The source workbook also contains national, provincial, city, municipality and unemployment-count observations, which are intentionally excluded.

In [0]:
WITH bronze AS (
 SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year, UPPER(TRIM(geographic_name)) AS region_name
 FROM pondolens.dev_bronze.unemployment_rate
 WHERE LOWER(TRIM(geographic_level))='region' AND LOWER(TRIM(indicator))='unemployment rate'
 AND TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) BETWEEN 2021 AND 2025
), counts AS (
 SELECT (SELECT COUNT(*) FROM bronze) AS bronze_in_scope_rows,
 (SELECT COUNT(*) FROM pondolens.dev_silver.unemployment_rate) AS silver_rows,
 (SELECT COUNT(DISTINCT CONCAT(CAST(year AS STRING),'|',region_name)) FROM bronze) AS bronze_distinct_name_year_keys
)
SELECT *, CASE WHEN bronze_distinct_name_year_keys = silver_rows THEN 'MATCH' ELSE 'REVIEW_MAPPING_OR_FILTERS' END AS status FROM counts;

## 6. Check required values and valid ranges
Expected null counts: zero. Unemployment rate is a percentage between 0 and 100. Review missing source values rather than treating them as zero.

In [0]:
SELECT COUNT(*) AS total_rows,
 COUNT_IF(unemployment_rate_id IS NULL) AS null_id, COUNT_IF(year IS NULL) AS null_year,
 COUNT_IF(region_code IS NULL) AS null_region_code, COUNT_IF(region_name IS NULL) AS null_region_name,
 COUNT_IF(unemployment_rate IS NULL) AS null_unemployment_rate,
 COUNT_IF(run_id IS NULL) AS null_run_id, COUNT_IF(source IS NULL) AS null_source,
 COUNT_IF(processed_at IS NULL) AS null_processed_at
FROM pondolens.dev_silver.unemployment_rate;
SELECT * FROM pondolens.dev_silver.unemployment_rate
WHERE unemployment_rate_id IS NULL OR unemployment_rate_id <= 100000000
OR year NOT BETWEEN 2021 AND 2025 OR region_code IS NULL OR TRIM(region_code)=''
OR region_name IS NULL OR TRIM(region_name)=''
OR unemployment_rate IS NULL OR unemployment_rate NOT BETWEEN 0 AND 100
OR run_id IS NULL OR source IS NULL OR processed_at IS NULL;

## 7. Check duplicate keys
Both queries should return zero rows. Grain is **one region per year** after selecting one source release.

In [0]:
SELECT unemployment_rate_id, COUNT(*) AS row_count FROM pondolens.dev_silver.unemployment_rate
GROUP BY unemployment_rate_id HAVING COUNT(*)>1;
SELECT year, region_code, COUNT(*) AS row_count FROM pondolens.dev_silver.unemployment_rate
GROUP BY year, region_code HAVING COUNT(*)>1;

## 8. Compare transformed source records
Expected: zero discrepancies. This re-applies the same selection rule and compares the resulting business columns using `EXCEPT ALL`. This is a Bronze-to-Silver transformation check, **not** independent Excel reconciliation.

In [0]:
WITH normalized AS (
 SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year,
        UPPER(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(region), ''), geographic_name), '\s+', ' '))) AS raw_region_name,
        TRY_CAST(REGEXP_REPLACE(TRIM(CAST(value AS STRING)), ',', '') AS DECIMAL(12,4)) AS unemployment_rate,
        LOWER(TRIM(estimate_status)) AS estimate_status,
        COALESCE(table_version, '') AS table_version
 FROM pondolens.dev_bronze.unemployment_rate
 WHERE TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) BETWEEN 2021 AND 2025
   AND LOWER(TRIM(geographic_level)) = 'region'
   AND LOWER(TRIM(indicator)) = 'unemployment rate'
), mapped AS (
 SELECT *,
   CASE
     WHEN raw_region_name RLIKE 'NATIONAL CAPITAL REGION|\(NCR\)' THEN 'NCR'
     WHEN raw_region_name RLIKE 'CORDILLERA ADMINISTRATIVE REGION|\(CAR\)' THEN 'CAR'
     WHEN raw_region_name RLIKE 'ILOCOS|REGION I \(' THEN 'I'
     WHEN raw_region_name RLIKE 'CAGAYAN VALLEY|REGION II \(' THEN 'II'
     WHEN raw_region_name RLIKE 'CENTRAL LUZON|REGION III \(' THEN 'III'
     WHEN raw_region_name RLIKE 'CALABARZON|REGION IV-A|REGION IVA' THEN 'IV-A'
     WHEN raw_region_name RLIKE 'MIMAROPA|REGION IV-B|REGION IVB' THEN 'IV-B'
     WHEN raw_region_name RLIKE 'BICOL|REGION V \(' THEN 'V'
     WHEN raw_region_name RLIKE 'WESTERN VISAYAS|REGION VI \(' THEN 'VI'
     WHEN raw_region_name RLIKE 'CENTRAL VISAYAS|REGION VII \(' THEN 'VII'
     WHEN raw_region_name RLIKE 'EASTERN VISAYAS|REGION VIII \(' THEN 'VIII'
     WHEN raw_region_name RLIKE 'ZAMBOANGA PENINSULA|REGION IX \(' THEN 'IX'
     WHEN raw_region_name RLIKE 'NORTHERN MINDANAO|REGION X \(' THEN 'X'
     WHEN raw_region_name RLIKE 'DAVAO|REGION XI \(' THEN 'XI'
     WHEN raw_region_name RLIKE 'SOCCSKSARGEN|REGION XII \(' THEN 'XII'
     WHEN raw_region_name RLIKE 'CARAGA|REGION XIII \(' THEN 'XIII'
     WHEN raw_region_name RLIKE 'BANGSAMORO|BARMM|ARMM|AUTONOMOUS REGION IN MUSLIM MINDANAO' THEN 'BARMM'
     WHEN raw_region_name RLIKE 'NEGROS ISLAND REGION|\(NIR\)' THEN 'NIR'
   END AS region_code
 FROM normalized
), valid AS (
 SELECT *,
 CASE region_code
 WHEN 'NCR' THEN 'National Capital Region' WHEN 'CAR' THEN 'Cordillera Administrative Region'
 WHEN 'I' THEN 'Ilocos Region' WHEN 'II' THEN 'Cagayan Valley' WHEN 'III' THEN 'Central Luzon'
 WHEN 'IV-A' THEN 'CALABARZON' WHEN 'IV-B' THEN 'MIMAROPA' WHEN 'V' THEN 'Bicol Region'
 WHEN 'VI' THEN 'Western Visayas' WHEN 'VII' THEN 'Central Visayas' WHEN 'VIII' THEN 'Eastern Visayas'
 WHEN 'IX' THEN 'Zamboanga Peninsula' WHEN 'X' THEN 'Northern Mindanao' WHEN 'XI' THEN 'Davao Region'
 WHEN 'XII' THEN 'SOCCSKSARGEN' WHEN 'XIII' THEN 'Caraga' WHEN 'BARMM' THEN 'BARMM'
 WHEN 'NIR' THEN 'Negros Island Region' END AS region_name,
 COALESCE(TRY_CAST(REGEXP_EXTRACT(table_version, '(20[0-9]{2})(?!.*20[0-9]{2})', 1) AS INT), 0) AS release_year
 FROM mapped WHERE region_code IS NOT NULL AND unemployment_rate BETWEEN 0 AND 100
), ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY year, region_code
 ORDER BY CASE estimate_status WHEN 'final' THEN 0 WHEN 'preliminary' THEN 1 ELSE 2 END,
 release_year DESC, unemployment_rate DESC, table_version DESC) AS selection_rank
 FROM valid
), expected AS (SELECT year, region_code, region_name, unemployment_rate FROM ranked WHERE selection_rank=1),
actual AS (SELECT year, region_code, region_name, unemployment_rate FROM pondolens.dev_silver.unemployment_rate),
missing AS (SELECT * FROM expected EXCEPT ALL SELECT * FROM actual),
extra AS (SELECT * FROM actual EXCEPT ALL SELECT * FROM expected)
SELECT 'MISSING_OR_CHANGED' AS issue, * FROM missing
UNION ALL SELECT 'EXTRA_OR_CHANGED' AS issue, * FROM extra;

## 9. Review geographic coverage and exclusions
Region counts can change over time (including NIR). Do not force a fixed 18-region count. Check unmapped/invalid regional rates before approving the build.

In [0]:
SELECT year, COUNT(*) AS total_rows, COUNT(DISTINCT region_code) AS distinct_regions,
 MIN(unemployment_rate) AS min_rate, MAX(unemployment_rate) AS max_rate
FROM pondolens.dev_silver.unemployment_rate GROUP BY year ORDER BY year;

In [0]:
WITH normalized AS (
 SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year,
        UPPER(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(region), ''), geographic_name), '\s+', ' '))) AS raw_region_name,
        TRY_CAST(REGEXP_REPLACE(TRIM(CAST(value AS STRING)), ',', '') AS DECIMAL(12,4)) AS unemployment_rate,
        LOWER(TRIM(estimate_status)) AS estimate_status,
        COALESCE(table_version, '') AS table_version
 FROM pondolens.dev_bronze.unemployment_rate
 WHERE TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) BETWEEN 2021 AND 2025
   AND LOWER(TRIM(geographic_level)) = 'region'
   AND LOWER(TRIM(indicator)) = 'unemployment rate'
), mapped AS (
 SELECT *,
   CASE
     WHEN raw_region_name RLIKE 'NATIONAL CAPITAL REGION|\(NCR\)' THEN 'NCR'
     WHEN raw_region_name RLIKE 'CORDILLERA ADMINISTRATIVE REGION|\(CAR\)' THEN 'CAR'
     WHEN raw_region_name RLIKE 'ILOCOS|REGION I \(' THEN 'I'
     WHEN raw_region_name RLIKE 'CAGAYAN VALLEY|REGION II \(' THEN 'II'
     WHEN raw_region_name RLIKE 'CENTRAL LUZON|REGION III \(' THEN 'III'
     WHEN raw_region_name RLIKE 'CALABARZON|REGION IV-A|REGION IVA' THEN 'IV-A'
     WHEN raw_region_name RLIKE 'MIMAROPA|REGION IV-B|REGION IVB' THEN 'IV-B'
     WHEN raw_region_name RLIKE 'BICOL|REGION V \(' THEN 'V'
     WHEN raw_region_name RLIKE 'WESTERN VISAYAS|REGION VI \(' THEN 'VI'
     WHEN raw_region_name RLIKE 'CENTRAL VISAYAS|REGION VII \(' THEN 'VII'
     WHEN raw_region_name RLIKE 'EASTERN VISAYAS|REGION VIII \(' THEN 'VIII'
     WHEN raw_region_name RLIKE 'ZAMBOANGA PENINSULA|REGION IX \(' THEN 'IX'
     WHEN raw_region_name RLIKE 'NORTHERN MINDANAO|REGION X \(' THEN 'X'
     WHEN raw_region_name RLIKE 'DAVAO|REGION XI \(' THEN 'XI'
     WHEN raw_region_name RLIKE 'SOCCSKSARGEN|REGION XII \(' THEN 'XII'
     WHEN raw_region_name RLIKE 'CARAGA|REGION XIII \(' THEN 'XIII'
     WHEN raw_region_name RLIKE 'BANGSAMORO|BARMM|ARMM|AUTONOMOUS REGION IN MUSLIM MINDANAO' THEN 'BARMM'
     WHEN raw_region_name RLIKE 'NEGROS ISLAND REGION|\(NIR\)' THEN 'NIR'
   END AS region_code
 FROM normalized
) SELECT year, raw_region_name, region_code, unemployment_rate, estimate_status, table_version
FROM mapped WHERE region_code IS NULL OR unemployment_rate IS NULL OR unemployment_rate NOT BETWEEN 0 AND 100 ORDER BY year, raw_region_name;

### Assumptions and open checks
- The source workbook contains overlapping releases; `Final` is preferred to `Preliminary`, followed by latest release year. Confirm this precedence with your team and inspect conflicting values.
- Bronze column names are assumed to match the supplied workbook. If Bronze already has `fiscal_year`, `region`, or `unemployment_rate` instead, adjust the input CTE.
- This table deliberately omits national and subregional geographies, unemployment counts, standard errors and estimate statuses because the requested Silver schema has no columns for them. Retain these in Bronze.
- Historical ARMM/BARMM names are standardized to BARMM; NIR is not backfilled into years when absent. Confirm mapping decisions against your canonical region dimension.
- Rows with unmapped regions or invalid/missing numeric rates are excluded and must be investigated.
- This is an initial-load `CREATE TABLE IF NOT EXISTS` workflow. For rebuilds, use an explicitly approved replacement strategy; do not silently append duplicates.
- Databricks SQL notebook cells may need their language switched to SQL after import.